In [ ]:
from google.colab import files
import pandas as pd
import io

# This will prompt you to upload the file
uploaded = files.upload()

# Get the actual filename (in case it uploaded as 'netflix_titles (1).csv')
filename = list(uploaded.keys())[0]

# Load the dataset using the correct filename
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Display results
print(f"Successfully loaded: {filename}")
print("Dataset Shape:", df.shape)
df.head()

Saving netflix_titles (1).csv to netflix_titles (1).csv
Successfully loaded: netflix_titles (1).csv
Dataset Shape: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
# 1. Check for missing values before we do anything
print("--- Missing Values BEFORE Cleaning ---")
print(df.isnull().sum())

# 2. Handle missing categorical data by replacing them with placeholders
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Not Given')

# 3. Drop rows where critical data (like date or rating) is missing
# (This usually only removes a tiny fraction of the 8000+ rows)
df.dropna(subset=['date_added', 'rating', 'duration'], inplace=True)

# 4. Remove any duplicate rows
df.drop_duplicates(inplace=True)

# 5. Verify the cleaning worked
print("\n--- Missing Values AFTER Cleaning ---")
print(df.isnull().sum())

--- Missing Values BEFORE Cleaning ---
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

--- Missing Values AFTER Cleaning ---
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


In [ ]:
import warnings
warnings.filterwarnings('ignore') # Hides annoying pandas warnings

# 1. Convert 'date_added' to a proper datetime object
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

# Extract the year and month the content was added to Netflix
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

# 2. Extract numeric values from the 'duration' column (e.g., turning "90 min" into 90)
df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)

# 3. Create a "Content Length Category" for Movies
# We will categorize them as Short (< 90 mins), Medium (90-120 mins), or Long (> 120 mins)
def categorize_length(row):
    if row['type'] == 'Movie':
        if row['duration_num'] < 90:
            return 'Short Movie'
        elif 90 <= row['duration_num'] <= 120:
            return 'Medium Movie'
        else:
            return 'Long Movie'
    else:
        return 'TV Show' # We leave TV shows alone for this metric

df['Content Length Category'] = df.apply(categorize_length, axis=1)

print("Feature engineering complete! Here is a peek at the new columns:")
display(df[['title', 'type', 'duration', 'duration_num', 'Content Length Category', 'year_added']].head())

<>:12: SyntaxWarning: invalid escape sequence '\d'
<>:12: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-2160/107264446.py:12: SyntaxWarning: invalid escape sequence '\d'
  df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)


Feature engineering complete! Here is a peek at the new columns:


,title,type,duration,duration_num,Content Length Category,year_added
0,Dick Johnson Is Dead,Movie,90 min,90.0,Medium Movie,2021
1,Blood & Water,TV Show,2 Seasons,2.0,TV Show,2021
2,Ganglands,TV Show,1 Season,1.0,TV Show,2021
3,Jailbirds New Orleans,TV Show,1 Season,1.0,TV Show,2021
4,Kota Factory,TV Show,2 Seasons,2.0,TV Show,2021


In [ ]:
import pandas as pd

# 1. Convert 'date_added' to Datetime
# We removed .str.strip() to avoid the AttributeError.
# 'errors="coerce"' will safely turn any unreadable junk data into NaT (Not a Time).
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# 2. Convert 'release_year' to Datetime
# 'release_year' is typically a number (e.g., 2020), so formatting it as '%Y'
# converts it to a standard date (e.g., 2020-01-01).
df['release_date'] = pd.to_datetime(df['release_year'], format='%Y', errors='coerce')

# 3. Verify the changes
# You should see 'datetime64[ns]' listed next to these columns now.
print("--- Data Types After Conversion ---")
print(df[['date_added', 'release_date']].dtypes)

# 4. Preview the new data
print("\n--- First 5 Rows ---")
print(df[['title', 'date_added', 'release_year', 'release_date']].head())

--- Data Types After Conversion ---
date_added      datetime64[ns]
release_date    datetime64[ns]
dtype: object

--- First 5 Rows ---
                   title date_added  release_year release_date
0   Dick Johnson Is Dead 2021-09-25          2020   2020-01-01
1          Blood & Water 2021-09-24          2021   2021-01-01
2              Ganglands 2021-09-24          2021   2021-01-01
3  Jailbirds New Orleans 2021-09-24          2021   2021-01-01
4           Kota Factory 2021-09-24          2021   2021-01-01
